In [ ]:
### This notebook contains the main codes used to develop the model. This notebook was large and I presented only the codes here which was written while working on the course project.

In [13]:
from __future__ import annotations

import os
import ast
import random
from typing import Any, Dict, Optional, Sequence, Tuple, List

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers


# ---------------------------------------------------------------------
# Global default
# ---------------------------------------------------------------------

RANDOM_STATE = 1234


# ---------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------

def set_tensorflow_reproducibility(random_state: int = RANDOM_STATE) -> None:
    """
    Set reproducibility controls for TensorFlow/Keras.

    Note:
    This improves reproducibility, but exact bit-level GPU reproducibility can still
    depend on CUDA, cuDNN, TensorFlow version, driver version, and hardware.
    """
    os.environ["PYTHONHASHSEED"] = str(random_state)

    random.seed(random_state)
    np.random.seed(random_state)
    tf.keras.utils.set_random_seed(random_state)

    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


# ---------------------------------------------------------------------
# Embedding parsing
# ---------------------------------------------------------------------

def embedding_value_to_1d_array(value: Any, *, row_index: Optional[Any] = None) -> np.ndarray:
    """
    Convert one dataframe embedding-cell value into a finite 1D float32 vector.

    Expected good formats:
        - np.ndarray
        - list / tuple of floats

    String parsing is supported only for simple list-like strings.
    Better: keep the dataframe column as real arrays/lists, not strings.
    """
    if isinstance(value, np.ndarray):
        arr = value

    elif isinstance(value, (list, tuple)):
        arr = np.asarray(value)

    elif isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            arr = np.asarray(parsed)
        except Exception as exc:
            raise ValueError(
                f"Embedding at row {row_index} is a string but could not be parsed. "
                "Store embeddings as arrays/lists instead of strings."
            ) from exc

    else:
        try:
            arr = np.asarray(value)
        except Exception as exc:
            raise ValueError(f"Could not convert embedding at row {row_index} to an array.") from exc

    arr = np.asarray(arr, dtype=np.float32).reshape(-1)

    if arr.size == 0:
        raise ValueError(f"Embedding at row {row_index} is empty.")

    if not np.all(np.isfinite(arr)):
        raise ValueError(f"Embedding at row {row_index} contains NaN or infinite values.")

    return arr


def dataframe_to_embedding_arrays(
    df: pd.DataFrame,
    *,
    embedding_col: str,
    label_col: Optional[str] = None,
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    """
    Convert dataframe embedding column into X matrix and optional y vector.

    Returns
    -------
    X: shape (n_samples, embedding_dim), float32
    y: shape (n_samples, 1), float32, or None
    """
    if embedding_col not in df.columns:
        raise ValueError(f"embedding_col='{embedding_col}' not found in dataframe.")

    embeddings: List[np.ndarray] = []

    for idx, value in df[embedding_col].items():
        embeddings.append(
            embedding_value_to_1d_array(value, row_index=idx)
        )

    dims = [arr.shape[0] for arr in embeddings]
    unique_dims = sorted(set(dims))

    if len(unique_dims) != 1:
        raise ValueError(f"Embeddings have inconsistent dimensions: {unique_dims}")

    X = np.vstack(embeddings).astype(np.float32)

    y = None

    if label_col is not None:
        if label_col not in df.columns:
            raise ValueError(f"label_col='{label_col}' not found in dataframe.")

        y_raw = df[label_col].astype(int).to_numpy()

        unique_labels = sorted(set(y_raw.tolist()))
        if not set(unique_labels).issubset({0, 1}):
            raise ValueError(f"Labels must be binary 0/1. Found labels: {unique_labels}")

        y = y_raw.astype(np.float32).reshape(-1, 1)

    return X, y


# ---------------------------------------------------------------------
# tf.data helpers
# ---------------------------------------------------------------------

def make_embedding_dataset_from_arrays(
    X: np.ndarray,
    y: Optional[np.ndarray] = None,
    *,
    batch_size: int = 128,
    shuffle: bool = False,
    random_state: int = RANDOM_STATE,
    cache: bool = False,
) -> tf.data.Dataset:
    """
    Build tf.data dataset from embedding arrays.

    Use shuffle=True only for training.
    Use shuffle=False for validation and test.
    """
    X = np.asarray(X, dtype=np.float32)

    if y is None:
        ds = tf.data.Dataset.from_tensor_slices(X)
    else:
        y = np.asarray(y, dtype=np.float32).reshape(-1, 1)
        ds = tf.data.Dataset.from_tensor_slices((X, y))

    if cache:
        ds = ds.cache()

    if shuffle:
        ds = ds.shuffle(
            buffer_size=len(X),
            seed=random_state,
            reshuffle_each_iteration=True,
        )

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return ds


def make_embedding_dataset_from_df(
    df: pd.DataFrame,
    *,
    embedding_col: str,
    label_col: Optional[str] = None,
    batch_size: int = 128,
    shuffle: bool = False,
    random_state: int = RANDOM_STATE,
    cache: bool = False,
    return_arrays: bool = False,
) -> Dict[str, Any]:
    """
    Convert dataframe to tf.data dataset.

    return_arrays=True returns:
        {
            "dataset": ds,
            "X": X,
            "y": y,
        }
    """
    X, y = dataframe_to_embedding_arrays(
        df,
        embedding_col=embedding_col,
        label_col=label_col,
    )

    ds = make_embedding_dataset_from_arrays(
        X,
        y,
        batch_size=batch_size,
        shuffle=shuffle,
        random_state=random_state,
        cache=cache,
    )

    out = {"dataset": ds}

    if return_arrays:
        out["X"] = X
        out["y"] = y

    return out


def make_test_embedding_dataset(
    test_df: pd.DataFrame,
    *,
    embedding_col: str,
    label_col: Optional[str] = None,
    batch_size: int = 128,
    cache: bool = False,
) -> Dict[str, Any]:
    """
    Process test dataframe into test dataset.

    Important:
    - No shuffle.
    - No fitting/adapting.
    - Original order is preserved.
    """
    return make_embedding_dataset_from_df(
        test_df,
        embedding_col=embedding_col,
        label_col=label_col,
        batch_size=batch_size,
        shuffle=False,
        random_state=RANDOM_STATE,
        cache=cache,
        return_arrays=True,
    )


# ---------------------------------------------------------------------
# Class imbalance
# ---------------------------------------------------------------------

def get_binary_class_weights_from_y(y_train: np.ndarray) -> Dict[int, float]:
    """
    Same class-weight idea as your existing code:
        weight_for_class = (1 / class_count) * (total / 2)
    """
    y = np.asarray(y_train).astype(int).reshape(-1)

    counts = np.bincount(y, minlength=2)
    neg = counts[0]
    pos = counts[1]

    if neg == 0 or pos == 0:
        raise ValueError(
            f"Training data must contain both classes. Found counts: class0={neg}, class1={pos}."
        )

    total = neg + pos

    return {
        0: float((1.0 / neg) * (total / 2.0)),
        1: float((1.0 / pos) * (total / 2.0)),
    }


@tf.keras.utils.register_keras_serializable()
class WeightedBinaryFocalLoss(tf.keras.losses.Loss):
    """
    Weighted binary focal loss for imbalanced binary classification.

    This is shape-safe:
    - y_true and y_pred are flattened to shape (batch,)
    - the returned loss is one scalar per sample
    - Keras handles final reduction
    """
    def __init__(
        self,
        w0: float,
        w1: float,
        gamma: float = 2.0,
        from_logits: bool = False,
        name: str = "weighted_binary_focal_loss",
        reduction: str = "sum_over_batch_size",
        **kwargs,
    ):
        super().__init__(name=name, reduction=reduction, **kwargs)
        self.w0 = float(w0)
        self.w1 = float(w1)
        self.gamma = float(gamma)
        self.from_logits = bool(from_logits)

    def call(self, y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.float32)
        y_pred = tf.cast(tf.reshape(y_pred, [-1]), tf.float32)

        if self.from_logits:
            y_prob = tf.sigmoid(y_pred)
        else:
            y_prob = y_pred

        eps = tf.keras.backend.epsilon()
        y_prob = tf.clip_by_value(y_prob, eps, 1.0 - eps)

        bce = -(
            y_true * tf.math.log(y_prob)
            + (1.0 - y_true) * tf.math.log(1.0 - y_prob)
        )

        p_t = y_true * y_prob + (1.0 - y_true) * (1.0 - y_prob)
        focal_factor = tf.pow(1.0 - p_t, self.gamma)

        class_weight = y_true * self.w1 + (1.0 - y_true) * self.w0

        return class_weight * focal_factor * bce

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "w0": self.w0,
                "w1": self.w1,
                "gamma": self.gamma,
                "from_logits": self.from_logits,
                "name": self.name,
            }
        )
        return config


# ---------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------

@tf.keras.utils.register_keras_serializable()
class BinarySpecificity(tf.keras.metrics.Metric):
    def __init__(self, threshold: float = 0.5, name: str = "specificity", **kwargs):
        super().__init__(name=name, **kwargs)
        self.threshold = float(threshold)
        self.tn = self.add_weight(name="tn", initializer="zeros")
        self.fp = self.add_weight(name="fp", initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true_bool = tf.cast(tf.reshape(y_true, [-1]), tf.bool)
        y_pred_bool = tf.greater_equal(tf.reshape(y_pred, [-1]), self.threshold)

        true_negative = tf.logical_and(
            tf.logical_not(y_true_bool),
            tf.logical_not(y_pred_bool),
        )
        false_positive = tf.logical_and(
            tf.logical_not(y_true_bool),
            y_pred_bool,
        )

        self.tn.assign_add(tf.reduce_sum(tf.cast(true_negative, tf.float32)))
        self.fp.assign_add(tf.reduce_sum(tf.cast(false_positive, tf.float32)))

    def result(self):
        return self.tn / (self.tn + self.fp + tf.keras.backend.epsilon())

    def reset_state(self):
        self.tn.assign(0.0)
        self.fp.assign(0.0)

    def get_config(self):
        config = super().get_config()
        config.update({"threshold": self.threshold})
        return config


@tf.keras.utils.register_keras_serializable()
class BalancedBinaryAccuracy(tf.keras.metrics.Metric):
    def __init__(self, threshold: float = 0.5, name: str = "balanced_accuracy", **kwargs):
        super().__init__(name=name, **kwargs)
        self.threshold = float(threshold)
        self.tp = self.add_weight(name="tp", initializer="zeros")
        self.tn = self.add_weight(name="tn", initializer="zeros")
        self.fp = self.add_weight(name="fp", initializer="zeros")
        self.fn = self.add_weight(name="fn", initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true_bool = tf.cast(tf.reshape(y_true, [-1]), tf.bool)
        y_pred_bool = tf.greater_equal(tf.reshape(y_pred, [-1]), self.threshold)

        tp = tf.logical_and(y_true_bool, y_pred_bool)
        tn = tf.logical_and(tf.logical_not(y_true_bool), tf.logical_not(y_pred_bool))
        fp = tf.logical_and(tf.logical_not(y_true_bool), y_pred_bool)
        fn = tf.logical_and(y_true_bool, tf.logical_not(y_pred_bool))

        self.tp.assign_add(tf.reduce_sum(tf.cast(tp, tf.float32)))
        self.tn.assign_add(tf.reduce_sum(tf.cast(tn, tf.float32)))
        self.fp.assign_add(tf.reduce_sum(tf.cast(fp, tf.float32)))
        self.fn.assign_add(tf.reduce_sum(tf.cast(fn, tf.float32)))

    def result(self):
        sensitivity = self.tp / (self.tp + self.fn + tf.keras.backend.epsilon())
        specificity = self.tn / (self.tn + self.fp + tf.keras.backend.epsilon())
        return 0.5 * (sensitivity + specificity)

    def reset_state(self):
        self.tp.assign(0.0)
        self.tn.assign(0.0)
        self.fp.assign(0.0)
        self.fn.assign(0.0)

    def get_config(self):
        config = super().get_config()
        config.update({"threshold": self.threshold})
        return config


def get_embedding_model_metrics(threshold: float = 0.5) -> List[tf.keras.metrics.Metric]:
    return [
        tf.keras.metrics.BinaryAccuracy(name="accuracy", threshold=threshold),
        tf.keras.metrics.Precision(name="precision", thresholds=threshold),
        tf.keras.metrics.Recall(name="sensitivity", thresholds=threshold),
        BinarySpecificity(name="specificity", threshold=threshold),
        BalancedBinaryAccuracy(name="balanced_accuracy", threshold=threshold),
        tf.keras.metrics.AUC(name="auroc", curve="ROC"),
        tf.keras.metrics.AUC(name="pr_auc", curve="PR"),
    ]


# ---------------------------------------------------------------------
# Model architecture
# ---------------------------------------------------------------------

def dense_embedding_block(
    x,
    *,
    units: int,
    dropout_rate: float,
    l2_strength: float,
    random_state: int,
    block_index: int,
):
    """
    Dense block for embedding classification.

    LayerNorm is used instead of BatchNorm because LayerNorm is per-sample and does not
    depend on batch statistics. This is a clean choice for embedding vectors.
    """
    x = layers.Dense(
        units,
        kernel_initializer="he_normal",
        kernel_regularizer=regularizers.l2(l2_strength),
        name=f"dense_{block_index}_{units}",
    )(x)
    x = layers.LayerNormalization(name=f"layernorm_{block_index}")(x)
    x = layers.Activation("gelu", name=f"gelu_{block_index}")(x)

    if dropout_rate > 0:
        x = layers.Dropout(
            dropout_rate,
            seed=random_state + block_index,
            name=f"dropout_{block_index}",
        )(x)

    return x


def build_fc_embedding_model(
    *,
    input_dim: int,
    normalization_layer: Optional[tf.keras.layers.Layer] = None,
    hidden_units: Sequence[int] = (512, 256, 128, 64),
    dropout_rates: Sequence[float] = (0.35, 0.30, 0.20, 0.10),
    l2_strength: float = 1e-4,
    gaussian_noise_std: float = 0.01,
    random_state: int = RANDOM_STATE,
) -> tf.keras.Model:
    """
    Fully connected binary classifier for Qwen embeddings.
    """
    if len(hidden_units) != len(dropout_rates):
        raise ValueError("hidden_units and dropout_rates must have the same length.")

    inputs = layers.Input(shape=(input_dim,), name="embedding_input")
    x = inputs

    if normalization_layer is not None:
        x = normalization_layer(x)

    if gaussian_noise_std > 0:
        x = layers.GaussianNoise(
            gaussian_noise_std,
            seed=random_state,
            name="embedding_gaussian_noise",
        )(x)

    for block_index, (units, dropout_rate) in enumerate(zip(hidden_units, dropout_rates), start=1):
        x = dense_embedding_block(
            x,
            units=int(units),
            dropout_rate=float(dropout_rate),
            l2_strength=float(l2_strength),
            random_state=random_state,
            block_index=block_index,
        )

    outputs = layers.Dense(
        1,
        activation="sigmoid",
        kernel_initializer="glorot_uniform",
        name="interaction_probability",
    )(x)

    model = models.Model(inputs=inputs, outputs=outputs, name="qwen_embedding_fc_interaction_model")

    return model


# ---------------------------------------------------------------------
# Threshold selection and reports
# ---------------------------------------------------------------------

def binary_metrics_from_arrays(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    *,
    threshold: float = 0.5,
) -> Dict[str, float]:
    y_true = np.asarray(y_true).astype(int).reshape(-1)
    y_prob = np.asarray(y_prob, dtype=float).reshape(-1)
    y_pred = (y_prob >= threshold).astype(int)

    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    f1 = 2 * precision * sensitivity / (precision + sensitivity) if np.isfinite(precision) and np.isfinite(sensitivity) and (precision + sensitivity) > 0 else np.nan
    balanced_accuracy = 0.5 * (sensitivity + specificity) if np.isfinite(sensitivity) and np.isfinite(specificity) else np.nan
    accuracy = (tp + tn) / len(y_true) if len(y_true) > 0 else np.nan

    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy),
        "balanced_accuracy": float(balanced_accuracy),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "precision": float(precision),
        "f1": float(f1),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


def select_best_threshold_on_validation(
    y_val: np.ndarray,
    y_val_prob: np.ndarray,
    *,
    thresholds: Optional[np.ndarray] = None,
) -> Dict[str, float]:
    """
    Select threshold using validation data only.

    Objective:
        maximize balanced accuracy.

    Tie-break:
        choose threshold closest to 0.5.
    """
    if thresholds is None:
        thresholds = np.linspace(0.05, 0.95, 181)

    rows = []

    for threshold in thresholds:
        metrics = binary_metrics_from_arrays(
            y_true=y_val,
            y_prob=y_val_prob,
            threshold=float(threshold),
        )
        rows.append(metrics)

    best = max(
        rows,
        key=lambda row: (
            row["balanced_accuracy"],
            -abs(row["threshold"] - 0.5),
        ),
    )

    return best


# ---------------------------------------------------------------------
# Training function
# ---------------------------------------------------------------------

def train_fc_embedding_model(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    *,
    embedding_col: str,
    label_col: str,
    batch_size: int = 128,
    epochs: int = 200,
    patience: int = 20,
    learning_rate: float = 3e-4,
    weight_decay: float = 1e-4,
    focal_gamma: float = 2.0,
    monitor: str = "val_loss",
    use_train_only_normalization: bool = True,
    hidden_units: Sequence[int] = (512, 256, 128, 64),
    dropout_rates: Sequence[float] = (0.35, 0.30, 0.20, 0.10),
    l2_strength: float = 1e-4,
    gaussian_noise_std: float = 0.01,
    random_state: int = RANDOM_STATE,
    verbose: int = 1,
) -> tf.keras.Model:
    """
    Train a fully connected network using only train_df and val_df.

    No leakage rules:
    - The input normalization layer is adapted on TRAIN embeddings only.
    - Class weights are computed from TRAIN labels only.
    - Early stopping monitors VALIDATION only.
    - Best threshold is selected from VALIDATION only.
    - Test data are never used here.

    Returns
    -------
    trained tf.keras.Model

    Extra attributes attached to model:
        model.best_threshold_
        model.validation_threshold_info_
        model.training_history_
        model.class_weight_
        model.embedding_col_
        model.label_col_
    """
    set_tensorflow_reproducibility(random_state)

    X_train, y_train = dataframe_to_embedding_arrays(
        train_df,
        embedding_col=embedding_col,
        label_col=label_col,
    )
    X_val, y_val = dataframe_to_embedding_arrays(
        val_df,
        embedding_col=embedding_col,
        label_col=label_col,
    )

    if X_train.shape[1] != X_val.shape[1]:
        raise ValueError(
            f"Train and validation embedding dimensions differ: "
            f"{X_train.shape[1]} vs {X_val.shape[1]}"
        )

    input_dim = int(X_train.shape[1])

    class_weight = get_binary_class_weights_from_y(y_train)

    train_ds = make_embedding_dataset_from_arrays(
        X_train,
        y_train,
        batch_size=batch_size,
        shuffle=True,
        random_state=random_state,
        cache=False,
    )

    val_ds = make_embedding_dataset_from_arrays(
        X_val,
        y_val,
        batch_size=batch_size,
        shuffle=False,
        random_state=random_state,
        cache=False,
    )

    normalization_layer = None

    if use_train_only_normalization:
        normalization_layer = layers.Normalization(
            axis=-1,
            name="train_only_embedding_normalization",
        )
        normalization_layer.adapt(X_train)

    model = build_fc_embedding_model(
        input_dim=input_dim,
        normalization_layer=normalization_layer,
        hidden_units=hidden_units,
        dropout_rates=dropout_rates,
        l2_strength=l2_strength,
        gaussian_noise_std=gaussian_noise_std,
        random_state=random_state,
    )

    loss = WeightedBinaryFocalLoss(
        w0=class_weight[0],
        w1=class_weight[1],
        gamma=focal_gamma,
        from_logits=False,
    )

    try:
        optimizer = tf.keras.optimizers.AdamW(
            learning_rate=learning_rate,
            weight_decay=weight_decay,
            clipnorm=1.0,
        )
    except Exception:
        optimizer = tf.keras.optimizers.Adam(
            learning_rate=learning_rate,
            clipnorm=1.0,
        )

    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=get_embedding_model_metrics(threshold=0.5),
    )

    mode = "min" if "loss" in monitor.lower() else "max"

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor=monitor,
            mode=mode,
            patience=patience,
            restore_best_weights=True,
            min_delta=1e-4,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor=monitor,
            mode=mode,
            factor=0.5,
            patience=max(3, patience // 3),
            min_lr=1e-6,
            verbose=1,
        ),
    ]

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=callbacks,
        verbose=verbose,
    )

    # Validation-only threshold selection.
    y_val_prob = model.predict(
        X_val,
        batch_size=batch_size,
        verbose=0,
    ).reshape(-1)

    threshold_info = select_best_threshold_on_validation(
        y_val=y_val.reshape(-1),
        y_val_prob=y_val_prob,
    )

    model.best_threshold_ = float(threshold_info["threshold"])
    model.validation_threshold_info_ = threshold_info
    model.training_history_ = history.history
    model.class_weight_ = class_weight
    model.embedding_col_ = embedding_col
    model.label_col_ = label_col
    model.random_state_ = random_state

    print("Training class weights:", class_weight)
    print("Validation-selected threshold:", model.best_threshold_)
    print("Validation threshold metrics:", threshold_info)

    return model


# ---------------------------------------------------------------------
# Test prediction
# ---------------------------------------------------------------------

def predict_on_embedding_df(
    model: tf.keras.Model,
    test_df: pd.DataFrame,
    *,
    embedding_col: str,
    label_col: Optional[str] = None,
    batch_size: int = 128,
    threshold: Optional[float] = None,
    proba_col: str = "predicted_proba",
    pred_col: str = "predicted_class",
) -> Dict[str, Any]:
    """
    Predict probabilities/classes on a test dataframe.

    Important:
    - No shuffle.
    - No fitting/adapting.
    - Original dataframe order is preserved.
    """
    X_test, y_test = dataframe_to_embedding_arrays(
        test_df,
        embedding_col=embedding_col,
        label_col=label_col,
    )

    y_prob = model.predict(
        X_test,
        batch_size=batch_size,
        verbose=0,
    ).reshape(-1)

    if threshold is None:
        threshold = getattr(model, "best_threshold_", 0.5)

    y_pred = (y_prob > threshold).astype(int)

    out_df = test_df.copy()
    out_df[proba_col] = y_prob
    out_df[pred_col] = y_pred

    metrics = None

    if y_test is not None:
        metrics = binary_metrics_from_arrays(
            y_true=y_test.reshape(-1),
            y_prob=y_prob,
            threshold=float(threshold),
        )

    return {
        "df": out_df,
        "X": X_test,
        "y": y_test,
        "predicted_proba": y_prob,
        "predicted_class": y_pred,
        "threshold": float(threshold),
        "metrics": metrics,
    }


# ---------------------------------------------------------------------
# Optional: save history
# ---------------------------------------------------------------------

def save_training_history_from_model(
    model: tf.keras.Model,
    output_path: str,
) -> None:
    """
    Save model.training_history_ if available.
    """
    if not hasattr(model, "training_history_"):
        raise ValueError("Model does not have training_history_ attribute.")

    pd.DataFrame(model.training_history_).to_pickle(output_path)

## Main

In [14]:
# Checked: 1
# def show_stats_on_train_data(df_temp_train, val_pid_list):
#     dict_train_count = dict(Counter(df_temp_train[str_category].tolist()))
#     train_1_to_0_class_ratio = dict_train_count.get('1') * 100 / dict_train_count.get('0')
#     print(f'train_class_ratio_1_to_0: {train_1_to_0_class_ratio}, val_pid_list: {val_pid_list}')


def start_developing_models():
    df_prediction = pd.DataFrame()

    df_image_path = pd.read_pickle('embeddings_cat_map.pkl')
    df_image_path[str_category] = df_image_path[str_category].astype(int)

    print(df_image_path.columns)

    list_pid = sorted(df_image_path[str_parti_id].unique().tolist())
    iteration_counter = 0

    for pid_index in range(0, len(list_pid), chunk_size_for_test):
        iteration_counter += 1
        test_pid_list = list_pid[pid_index : pid_index + chunk_size_for_test]
        loc_model_file = os.path.join(loc_models, 'iter_'+ str(pid_index)+'__'+ test_pid_list[0]+'__chunk_test_'+str(chunk_size_for_test) +'_val_'+ str(chunk_size_for_val) +'_'+ special_condition +'.keras')

        
        print('Started ', pid_index, ' iter_counter: ', iteration_counter, test_pid_list)

        ########### Seperating Training, Test, Validation Data
        mask_test_ds = df_image_path[str_parti_id].isin(test_pid_list)
        df_test = df_image_path[ mask_test_ds ].copy()
        df_temp_train = df_image_path[~ mask_test_ds ].copy()

        val_pid_list = get_list_pid_for_val(df= df_temp_train, col_pid= str_parti_id, col_label= str_category)
        mask_val_ds = df_temp_train[str_parti_id].isin(val_pid_list)

        df_val = df_temp_train[ mask_val_ds ].copy()
        df_temp_train = df_temp_train[~ mask_val_ds].copy()
        # show_stats_on_train_data(df_temp_train, val_pid_list)

        df_temp_train = shuffle(df_temp_train, random_state=random_state).reset_index(drop=True)
        df_val = shuffle(df_val, random_state=random_state).reset_index(drop=True)
        df_test = shuffle(df_test, random_state=random_state).reset_index(drop=True)
        print('Classes in training dataset: ', Counter(df_temp_train[str_category]), 'Classes in validation dataset: ', Counter(df_val[str_category]), 'Classes in test dataset: ', Counter(df_test[str_category]))

        model = train_fc_embedding_model(
                                        train_df=df_temp_train,
                                        val_df=df_val,
                                        embedding_col="embedding",
                                        label_col= str_category,
                                        batch_size=48,
                                        learning_rate=0.003,
                                        epochs=50,
                                        patience=10,
                                        random_state= random_state)
        

        pred_pack = predict_on_embedding_df(
                                    model=model,
                                    test_df=df_test,
                                    embedding_col="embedding",
                                    label_col=str_category,
                                    batch_size=128,
                                )

        # df_test_predictions = pred_pack["df"]
        # test_metrics = pred_pack["predicted_class"]
        
        
        df_test[str_predicted_proba] = pred_pack["predicted_proba"]
        df_test[str_predicted_class] = pred_pack["predicted_class"]
        df_prediction =  pd.concat([df_prediction, df_test])
        df_prediction[str_category] = df_prediction[str_category].astype(int)

        print(iteration_counter, datetime.now(), '# of samples:', df_prediction.shape[0], clf_report(true_labels= df_prediction[str_category].tolist(), predicted_val_list= df_prediction[str_predicted_class].tolist(), bool_print=False))
        df_prediction.to_excel(os.path.join(loc_performances, special_condition+ '__models_performances.xlsx'), index=False)

        K.clear_session()
        gc.collect()
    
    os._exit(00) # to ensure reproducibility, we need to run from begining and hence, set it. [to be honest, I am still confused why we need to restart from begining to make the findings very close or very precisely reproducible. Perhaps, some memory is consumed by the noetbook somehow if we run the model. Maybe, even stopping does not release that memory.]

start_developing_models()

Index(['Audio', 'category', 'P_ID', 'T', 'embedding', 'metadata',
       'accelerometer_caption', 'gravity_caption', 'ppg_caption',
       'light_caption', 'step_caption', 'merged_caption'],
      dtype='object')
Started  0  iter_counter:  1 ['PA01']
just checking val pid list list_pid_alread_added []
just checking val pid list list_pid_alread_added ['PA11']
Classes in training dataset:  Counter({0: 21396, 1: 9792}) Classes in validation dataset:  Counter({1: 603, 0: 569}) Classes in test dataset:  Counter({0: 1104, 1: 199})
Epoch 1/50
650/650 ━━━━━━━━━━━━━━━━━━━━ 26s 25ms/step - accuracy: 0.5424 - auroc: 0.5767 - balanced_accuracy: 0.5583 - loss: 0.2634 - pr_auc: 0.3622 - precision: 0.3622 - sensitivity: 0.6012 - specificity: 0.5154 - val_accuracy: 0.5614 - val_auroc: 0.5651 - val_balanced_accuracy: 0.5549 - val_loss: 0.2184 - val_pr_auc: 0.5482 - val_precision: 0.5522 - val_sensitivity: 0.7811 - val_specificity: 0.3286 - learning_rate: 0.0030
Epoch 2/50
650/650 ━━━━━━━━━━━━━━━━━━━━ 1

E0000 00:00:1778663113.000170 1919361 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


Training class weights: {0: 0.7288278182837913, 1: 1.5925245098039216}
Validation-selected threshold: 0.5049999999999999
Validation threshold metrics: {'threshold': 0.5049999999999999, 'accuracy': 0.5930034129692833, 'balanced_accuracy': 0.5862937800744374, 'sensitivity': 0.8175787728026535, 'specificity': 0.35500878734622143, 'precision': 0.5732558139534883, 'f1': 0.6739576213260424, 'tp': 493, 'tn': 202, 'fp': 367, 'fn': 110}
1 2026-05-13 05:05:13.750760 # of samples: 1303 [52.45, 72.36, 36.23, 54.3, 39.41, 41.75]
Started  1  iter_counter:  2 ['PA02']
just checking val pid list list_pid_alread_added []
just checking val pid list list_pid_alread_added ['PA11']
Classes in training dataset:  Counter({0: 21824, 1: 9843}) Classes in validation dataset:  Counter({1: 603, 0: 569}) Classes in test dataset:  Counter({0: 676, 1: 148})
Epoch 1/50
660/660 ━━━━━━━━━━━━━━━━━━━━ 21s 25ms/step - accuracy: 0.5462 - auroc: 0.5782 - balanced_accuracy: 0.5583 - loss: 0.2563 - pr_auc: 0.3598 - precision:

E0000 00:00:1778663620.212005 1919361 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


Training class weights: {0: 0.7255086143695014, 1: 1.6086051000711166}
Validation-selected threshold: 0.5099999999999999
Validation threshold metrics: {'threshold': 0.5099999999999999, 'accuracy': 0.5767918088737202, 'balanced_accuracy': 0.5707374084469277, 'sensitivity': 0.7794361525704809, 'specificity': 0.36203866432337434, 'precision': 0.5642256902761105, 'f1': 0.6545961002785515, 'tp': 470, 'tn': 206, 'fp': 363, 'fn': 133}
2 2026-05-13 05:13:40.838096 # of samples: 2127 [52.42, 66.57, 41.97, 54.27, 42.6, 45.98]
Started  2  iter_counter:  3 ['PA03']
just checking val pid list list_pid_alread_added []
just checking val pid list list_pid_alread_added ['PA11']
Classes in training dataset:  Counter({0: 21688, 1: 9652}) Classes in validation dataset:  Counter({1: 603, 0: 569}) Classes in test dataset:  Counter({0: 812, 1: 339})
Epoch 1/50
653/653 ━━━━━━━━━━━━━━━━━━━━ 21s 25ms/step - accuracy: 0.5434 - auroc: 0.5762 - balanced_accuracy: 0.5588 - loss: 0.2586 - pr_auc: 0.3562 - precision:

...

E0000 00:00:1778681692.742035 1919361 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


Training class weights: {0: 0.7255820360931258, 1: 1.60824427480916}
Validation-selected threshold: 0.48999999999999994
Validation threshold metrics: {'threshold': 0.48999999999999994, 'accuracy': 0.5802047781569966, 'balanced_accuracy': 0.570734493904234, 'sensitivity': 0.8971807628524047, 'specificity': 0.24428822495606328, 'precision': 0.557157569515963, 'f1': 0.6874205844980941, 'tp': 541, 'tn': 139, 'fp': 430, 'fn': 62}
37 2026-05-13 10:14:53.424391 # of samples: 32717 [57.19, 78.52, 35.71, 57.11, 49.33, 49.34]
Started  37  iter_counter:  38 ['PB18']
just checking val pid list list_pid_alread_added []
just checking val pid list list_pid_alread_added ['PA11']
Classes in training dataset:  Counter({0: 21734, 1: 9811}) Classes in validation dataset:  Counter({1: 603, 0: 569}) Classes in test dataset:  Counter({0: 766, 1: 180})
Epoch 1/50
658/658 ━━━━━━━━━━━━━━━━━━━━ 22s 26ms/step - accuracy: 0.5395 - auroc: 0.5743 - balanced_accuracy: 0.5542 - loss: 0.2614 - pr_auc: 0.3584 - precisio

E0000 00:00:1778682202.569598 1919361 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


Training class weights: {0: 0.7257062666789362, 1: 1.6076342880440322}
Validation-selected threshold: 0.48499999999999993
Validation threshold metrics: {'threshold': 0.48499999999999993, 'accuracy': 0.6083617747440273, 'balanced_accuracy': 0.5991877169512718, 'sensitivity': 0.9154228855721394, 'specificity': 0.28295254833040423, 'precision': 0.575, 'f1': 0.7063339731285988, 'tp': 552, 'tn': 161, 'fp': 408, 'fn': 51}
38 2026-05-13 10:23:23.293398 # of samples: 33663 [57.27, 78.75, 35.71, 57.23, 49.26, 49.26]


: 